# Clasificación de Géneros de Películas

### Parte 3: Modelado

---

Comparación de 9 estrategias de clasificación multi-etiqueta. Métrica: **ROC-AUC macro** — un clasificador binario independiente por cada uno de los 24 géneros (`OneVsRestClassifier`).

In [ ]:
# Librerías
import warnings
warnings.filterwarnings('ignore')
import re, ast
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
sns.set_theme(style='whitegrid')

In [ ]:
# Cargar datos
dataTraining = pd.read_csv('data/dataTraining.csv', encoding='UTF-8', index_col=0)
dataTesting  = pd.read_csv('data/dataTesting.csv',  encoding='UTF-8', index_col=0)

# --- Pipeline de preprocesamiento (ver Parte 2) ---
dataTraining['genres'] = dataTraining['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
dataTraining['genres'] = dataTraining['genres'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
dataTraining = dataTraining.drop_duplicates()
dataTraining['genres'] = dataTraining['genres'].apply(lambda x: list(x) if isinstance(x, tuple) else x)

stop_words = set(stopwords.words('english'))

def remove_tags(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[\W_]', ' ', text)
    return text.lower()

def remove_stopwords(text):
    return ' '.join(w for w in text.split() if w not in stop_words)

w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer  = nltk.stem.WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join(lemmatizer.lemmatize(w) for w in w_tokenizer.tokenize(text))

# Aplicar pipeline a entrenamiento y test
for df in [dataTraining, dataTesting]:
    df['plot'] = df['plot'].apply(remove_tags).apply(remove_stopwords).apply(lemmatize_text)

# Codificar géneros como matriz binaria
mlb  = MultiLabelBinarizer()
y    = mlb.fit_transform(dataTraining['genres'])
cols = [f'p_{g}' for g in mlb.classes_]

# Split 80/20
X_train, X_val, y_train, y_val = train_test_split(
    dataTraining['plot'], y, test_size=0.2, random_state=42
)
print(f'Preprocesamiento listo. Train: {len(X_train)} | Val: {len(X_val)}')

In [ ]:
# Función reutilizable para graficar curvas ROC por clase
def plot_roc(y_true, y_pred, class_names, title='Curvas ROC'):
    fpr, tpr, roc_d = {}, {}, {}
    n = y_pred.shape[1]
    for i in range(n):
        fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_pred[:, i])
        roc_d[i] = auc(fpr[i], tpr[i])
    # Curva macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n)]))
    mean_tpr = sum(np.interp(all_fpr, fpr[i], tpr[i]) for i in range(n)) / n
    macro = auc(all_fpr, mean_tpr)
    # Gráfico
    plt.figure(figsize=(11, 8))
    plt.plot(all_fpr, mean_tpr, 'navy', lw=3, linestyle=':',
             label=f'Macro-average (AUC={macro:.3f})')
    for i in range(n):
        plt.plot(fpr[i], tpr[i], lw=1, label=f'{class_names[i]} ({roc_d[i]:.2f})')
    plt.plot([0,1],[0,1],'--', color='gray', lw=1.5)
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(title)
    plt.legend(loc='lower right', fontsize='x-small', ncol=2)
    plt.tight_layout(); plt.show()
    return roc_d

### 1. TF-IDF + Logistic Regression (baseline)

In [ ]:
# Pipeline: vectorización TF-IDF + clasificador logístico OneVsRest
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', OneVsRestClassifier(
        LogisticRegression(class_weight='balanced', random_state=42, n_jobs=-1)
    ))
])

# Grilla de hiperparámetros: n-gramas, vocabulario, regularización, solver
param_grid_lr = {
    'clf__estimator__max_iter': [100, 200],
    'clf__estimator__C': [0.01, 1, 5],
    'clf__estimator__solver': ['liblinear', 'saga'],
    'tfidf__ngram_range': [(1,1), (1,2), (1,3)],
    'tfidf__max_features': [20000, 25000],
}

# Búsqueda con validación cruzada (scoring=roc_auc promedia por clase en OneVsRest)
gs_lr = GridSearchCV(pipeline_lr, param_grid_lr, scoring='roc_auc', cv=3, n_jobs=-1, verbose=1)
gs_lr.fit(X_train, y_train)

y_pred_lr = gs_lr.predict_proba(X_val)
print(f'Mejores params: {gs_lr.best_params_}')
print(f'ROC-AUC macro (val): {roc_auc_score(y_val, y_pred_lr, average="macro"):.4f}')

In [ ]:
# Top 5 configuraciones del grid search
top5_lr = pd.DataFrame(gs_lr.cv_results_).sort_values('mean_test_score', ascending=False).head(5)

plt.figure(figsize=(10, 4))
bars = plt.barh(range(5), top5_lr['mean_test_score'], color='steelblue')
plt.yticks(range(5), [f'Config {i+1}' for i in range(5)])
plt.xlabel('ROC AUC (CV promedio)')
plt.title('LR baseline — top 5 configuraciones')
plt.gca().invert_yaxis()
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.001, bar.get_y() + bar.get_height()/2, f'{w:.4f}', va='center')
plt.tight_layout(); plt.show()

### 2. TF-IDF + Logistic Regression (trigramas + lematización) ⭐

In [ ]:
# Mejor configuración identificada: trigramas, 20k features, C=1, liblinear
tfidf_champ = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,3))
X_tfidf = tfidf_champ.fit_transform(dataTraining['plot'])

# Split 67/33 para evaluación final del modelo campeón
X_tr, X_te, y_tr, y_te = train_test_split(X_tfidf, y, test_size=0.33, random_state=42)

clf_champ = OneVsRestClassifier(
    LogisticRegression(max_iter=100, C=1, penalty='l2', solver='liblinear',
                       random_state=42, n_jobs=-1)
)
clf_champ.fit(X_tr, y_tr)

y_pred_champ = clf_champ.predict_proba(X_te)
print(f'ROC-AUC macro (test): {roc_auc_score(y_te, y_pred_champ, average="macro"):.4f}')

In [ ]:
# Curvas ROC por género para el modelo campeón
plot_roc(y_te, y_pred_champ, mlb.classes_, 'Modelo campeón — ROC por género')

### 3. Random Forest

In [ ]:
# Random Forest sobre TF-IDF: peor que LR en espacios dispersos de alta dimensión
# Los árboles necesitan muchos splits para aproximar la frontera lineal que LR captura directo
X_tr_rf, X_te_rf, y_tr_rf, y_te_rf = train_test_split(
    dataTraining['plot'], y, test_size=0.33, random_state=42
)

pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', OneVsRestClassifier(RandomForestClassifier(
        n_estimators=200, max_depth=50, min_samples_leaf=3,
        min_samples_split=10, random_state=42
    )))
])

pipeline_rf.fit(X_tr_rf, y_tr_rf)
y_pred_rf = pipeline_rf.predict_proba(X_te_rf)
print(f'ROC-AUC macro (test): {roc_auc_score(y_te_rf, y_pred_rf, average="macro"):.4f}')

### 4. Ridge Classifier

In [ ]:
# Ridge usa decision_function (no predict_proba) → score continuo válido para ROC-AUC
tfidf_ridge = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,3), norm='l2')
X_ridge = tfidf_ridge.fit_transform(dataTraining['plot'])

X_tr_ri, X_te_ri, y_tr_ri, y_te_ri = train_test_split(X_ridge, y, test_size=0.33, random_state=42)

clf_ridge = OneVsRestClassifier(RidgeClassifier(alpha=10.0, tol=1e-3, random_state=42))
clf_ridge.fit(X_tr_ri, y_tr_ri)

y_pred_ridge = clf_ridge.decision_function(X_te_ri)
print(f'ROC-AUC macro (test): {roc_auc_score(y_te_ri, y_pred_ridge, average="macro"):.4f}')

### 5. Feature Union (TF-IDF + longitud del plot)

In [ ]:
# Hipótesis: la longitud del plot agrega señal → no mejoró el AUC
# El género está en el contenido semántico, no en la extensión de la sinopsis
X_tr_fu, X_te_fu, y_tr_fu, y_te_fu = train_test_split(
    dataTraining['plot'], y, test_size=0.33, random_state=42
)

def text_length(X):
    # Feature adicional: número de palabras por sinopsis
    return np.array([len(t.split()) for t in X]).reshape(-1, 1)

combined = FeatureUnion([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,3))),
    ('length', FunctionTransformer(text_length, validate=False))
])

pipeline_fu = Pipeline([
    ('features', combined),
    ('clf', OneVsRestClassifier(
        LogisticRegression(C=1, solver='liblinear', penalty='l2', max_iter=100, n_jobs=-1)
    ))
])

pipeline_fu.fit(X_tr_fu, y_tr_fu)
y_pred_fu = pipeline_fu.predict_proba(X_te_fu)
print(f'ROC-AUC macro (test): {roc_auc_score(y_te_fu, y_pred_fu, average="macro"):.4f}')

### 6. Naive Bayes (ComplementNB)

In [ ]:
# ComplementNB: variante de Naive Bayes más robusta en clases desbalanceadas
# Entrena sobre el complemento de cada clase en lugar de la clase directa
X_tr_nb, X_te_nb, y_tr_nb, y_te_nb = train_test_split(
    dataTraining['plot'], y, test_size=0.33, random_state=42
)

pipeline_nb = Pipeline([
    ('count_vec', CountVectorizer(stop_words='english', max_features=20000, ngram_range=(1,3))),
    ('clf', OneVsRestClassifier(ComplementNB(alpha=0.1)))
])

pipeline_nb.fit(X_tr_nb, y_tr_nb)
y_pred_nb = pipeline_nb.predict_proba(X_te_nb)
print(f'ROC-AUC macro (test): {roc_auc_score(y_te_nb, y_pred_nb, average="macro"):.4f}')

### 7. LightGBM

In [ ]:
# LightGBM sobre TF-IDF: gradient boosting eficiente pero subóptimo en matrices dispersas
# Los árboles no explotan la linealidad del espacio bag-of-words
tfidf_lgbm = TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1,1))
X_lgbm = tfidf_lgbm.fit_transform(dataTraining['plot'])

X_tr_lg, X_te_lg, y_tr_lg, y_te_lg = train_test_split(X_lgbm, y, test_size=0.33, random_state=42)

clf_lgbm = OneVsRestClassifier(
    LGBMClassifier(objective='binary', n_estimators=100, learning_rate=0.1,
                   num_leaves=31, n_jobs=-1, random_state=42, verbose=-1)
)
clf_lgbm.fit(X_tr_lg, y_tr_lg)

y_pred_lgbm = clf_lgbm.predict_proba(X_te_lg)
print(f'ROC-AUC macro (test): {roc_auc_score(y_te_lg, y_pred_lgbm, average="macro"):.4f}')

### 8. Red Neuronal Densa (TF-IDF)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Vectorizar con TF-IDF (unigramas, 10k features — la red aprende combinaciones)
tfidf_nn = TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1,1))
X_nn = tfidf_nn.fit_transform(dataTraining['plot'])

X_tr_nn, X_te_nn, y_tr_nn, y_te_nn = train_test_split(X_nn, y, test_size=0.33, random_state=42)

# Arquitectura: 2 capas densas con dropout para regularización
model_nn = Sequential([
    Dense(256, input_shape=(X_tr_nn.shape[1],), activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_tr_nn.shape[1], activation='sigmoid')  # una salida por género
])

model_nn.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping para evitar overfitting
es = EarlyStopping(patience=3, restore_best_weights=True)
model_nn.fit(
    X_tr_nn.toarray(), y_tr_nn,
    validation_split=0.2, epochs=20, batch_size=64,
    callbacks=[es], verbose=1
)

y_pred_nn = model_nn.predict(X_te_nn.toarray())
print(f'ROC-AUC macro (test): {roc_auc_score(y_te_nn, y_pred_nn, average="macro"):.4f}')

### 9. BiLSTM + GloVe embeddings

In [ ]:
import os
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GlobalMaxPooling1D
from tensorflow.keras.metrics import AUC

GLOVE_PATH    = 'data/glove.6B.100d.txt'
EMBEDDING_DIM = 100
MAX_WORDS, MAX_LEN = 20000, 300

if not os.path.exists(GLOVE_PATH):
    # Instrucciones de descarga si el archivo no existe
    print('Descarga GloVe 6B desde: https://nlp.stanford.edu/projects/glove/')
    print('Coloca glove.6B.100d.txt en data/')
else:
    # Cargar vectores preentrenados de GloVe
    embedding_index = {}
    with open(GLOVE_PATH, encoding='utf8') as f:
        for line in f:
            values = line.split()
            embedding_index[values[0]] = np.asarray(values[1:], dtype='float32')
    print(f'Embeddings cargados: {len(embedding_index):,} palabras')

    # Tokenizar y truncar/rellenar secuencias a longitud fija
    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
    tokenizer.fit_on_texts(dataTraining['plot'])
    X_seq = pad_sequences(tokenizer.texts_to_sequences(dataTraining['plot']), maxlen=MAX_LEN)

    X_tr_gl, X_te_gl, y_tr_gl, y_te_gl = train_test_split(X_seq, y, test_size=0.2, random_state=42)

    # Construir matriz de embeddings con pesos GloVe preentrenados
    word_index = tokenizer.word_index
    num_words  = min(MAX_WORDS, len(word_index) + 1)
    embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
    for word, i in word_index.items():
        if i < num_words:
            vec = embedding_index.get(word)
            if vec is not None:
                embedding_matrix[i] = vec

    # Arquitectura BiLSTM: procesa la secuencia en ambas direcciones
    model_glove = Sequential([
        Embedding(num_words, EMBEDDING_DIM, weights=[embedding_matrix],
                  input_length=MAX_LEN, trainable=False),  # embeddings congelados
        Bidirectional(LSTM(64, return_sequences=True)),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(y.shape[1], activation='sigmoid')  # una salida por género
    ])

    model_glove.compile(
        optimizer='adam', loss='binary_crossentropy',
        metrics=[AUC(name='auc_macro', multi_label=True)]
    )

    es_glove = EarlyStopping(patience=2, restore_best_weights=True)
    model_glove.fit(X_tr_gl, y_tr_gl, validation_data=(X_te_gl, y_te_gl),
                    epochs=10, batch_size=64, callbacks=[es_glove], verbose=1)

    y_pred_glove = model_glove.predict(X_te_gl)
    print(f'ROC-AUC macro (test): {roc_auc_score(y_te_gl, y_pred_glove, average="macro"):.4f}')